In [1]:
import pandas as pd

ct_rate_df = pd.read_csv("/raid26/niraj/storage/ctrate_cache/updated_tag_extraction/train.csv")
segmed_df = pd.read_csv("/raid26/homagni/segmed_high_quality_ct/tag_extracted_cls/train.csv")
print(ct_rate_df.shape, segmed_df.shape)

(46988, 32) (211012, 37)


In [2]:
print(f"ct_rate columns: {ct_rate_df.columns}", f"segmed_df columns: {segmed_df.columns}")

ct_rate columns: Index(['series_uid', 'file_path', 'normal_study',
       'suspicious_pulmonary_lesion', 'suspicious_lymphadenopathy',
       'pleural_effusion', 'pleural_thickening_or_nodularity',
       'pericardial_effusion', 'airway_obstruction', 'postobstructive_change',
       'distant_metastasis_suspected', 'emphysema', 'bronchiectasis',
       'interstitial_lung_disease', 'pulmonary_edema', 'pneumonia',
       'diffuse_ground_glass_opacities', 'tree_in_bud', 'atelectasis',
       'scarring_or_fibrosis',
       'calcified_granuloma_or_old_granulomatous_disease',
       'perifissural_nodules', 'cardiomegaly', 'coronary_calcification',
       'aortic_atherosclerosis', 'mosaic_attenuation', 'bullae', 'lung_cysts',
       'known_malignancy', 'lung_nodule', 'masses', 'lymph_nodes'],
      dtype='object') segmed_df columns: Index(['series_uid', 'file_path', 'normal_study',
       'suspicious_pulmonary_lesion', 'suspicious_lymphadenopathy',
       'pleural_effusion', 'pleural_thickenin

In [19]:
voxtell_prompt_map = {
    # Lung lesion / malignancy-like findings
    "suspicious_pulmonary_lesion": "lung lesion",          # exact VoxTell vocab
    "lung_nodule": "lung nodule",                          # exact VoxTell vocab
    "masses": "lung tumor",                                # exact VoxTell vocab; closest for mass-like pulmonary lesion
    "known_malignancy": "lung tumor",                      # exact VoxTell vocab; use broader "tumor" if malignancy site is unknown
    "distant_metastasis_suspected": "metastatic lesion",    # generated VoxTell-style prompt

    # Lymph nodes
    "suspicious_lymphadenopathy": "mediastinal lymph nodes", # exact VoxTell vocab
    "lymph_nodes": "mediastinal lymph nodes",                # exact VoxTell vocab

    # Pleura
    "pleural_effusion": "pleural effusion",                # exact VoxTell vocab
    "pleural_thickening_or_nodularity": "pleura lesion",   # exact VoxTell vocab

    # Pericardium / heart / vascular
    "pericardial_effusion": "pericardial effusion",        # generated VoxTell-style prompt
    "cardiomegaly": "heart",                      # generated; better than "heart" for abnormality
    "coronary_calcification": "coronary artery calcification", # generated; paper has coronary artery + coronary calcification variants
    "aortic_atherosclerosis": "aortic atherosclerosis",    # generated; paper has aorta/aortic vessel prompts

    # Airway
    "airway_obstruction": "airway obstruction",            # generated; paper has "airways"
    "postobstructive_change": "postobstructive lung change", # generated
    "bronchiectasis": "bronchiectasis",                    # generated; paper has "bronchi"

    # Lung parenchymal / diffuse disease
    "emphysema": "pulmonary emphysema",                    # generated
    "interstitial_lung_disease": "interstitial lung disease", # generated
    "pulmonary_edema": "pulmonary edema",                  # generated
    "pneumonia": "pneumonia",                              # generated
    "diffuse_ground_glass_opacities": "diffuse ground glass opacities", # generated
    "tree_in_bud": "tree-in-bud opacities",                # generated
    "atelectasis": "atelectasis",                          # generated
    "rounded_atelectasis": "rounded atelectasis",          # generated
    "scarring_or_fibrosis": "fibrosis", # generated
    "calcified_granuloma_or_old_granulomatous_disease": (
        "calcified granuloma"
    ),                                                     # generated
    "perifissural_nodules": "perifissural lung nodules",   # generated; more specific than exact "lung nodule"
    "mosaic_attenuation": "mosaic attenuation",            # generated
    "bullae": "pulmonary bullae",                          # generated
    "lung_cysts": "lung cysts",                            # generated

    # SegMed-only acute air / embolic findings
    "pneumothorax": "pneumothorax",                        # generated
    "pneumomediastinum": "pneumomediastinum",              # generated
    "subcutaneous_emphysema": "subcutaneous emphysema",    # generated
    "pulmonary_embolism": "pulmonary embolism",            # generated; use "pulmonary artery" only if segmenting vessel
}

In [20]:
prompts = list(set(voxtell_prompt_map.values()))
print(f"Unique prompts ({len(prompts)}): {prompts}")

Unique prompts (32): ['lung tumor', 'pulmonary embolism', 'pleura lesion', 'pneumomediastinum', 'pulmonary emphysema', 'fibrosis', 'metastatic lesion', 'bronchiectasis', 'pericardial effusion', 'aortic atherosclerosis', 'heart', 'perifissural lung nodules', 'lung lesion', 'subcutaneous emphysema', 'airway obstruction', 'mediastinal lymph nodes', 'pleural effusion', 'pulmonary edema', 'lung nodule', 'tree-in-bud opacities', 'lung cysts', 'postobstructive lung change', 'coronary artery calcification', 'diffuse ground glass opacities', 'mosaic attenuation', 'pneumonia', 'pulmonary bullae', 'calcified granuloma', 'rounded atelectasis', 'atelectasis', 'pneumothorax', 'interstitial lung disease']


In [18]:
filepaths = ct_rate_df['file_path'].tolist() + segmed_df['file_path'].tolist()
print(len(filepaths))

258000


In [25]:
import random

random.shuffle(filepaths)

In [28]:
from pathlib import Path
import json

out_path = Path("/raid27/mohammad.fahim/voxtell_seg_masks/segmed_ct_rate_data.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w") as f:
    for img_path in filepaths:
        demo_record = {
            "image": img_path,
            "label": prompts,
            "modality": "CT",
            "orientation_code": "RAS",
        }
        f.write(json.dumps(demo_record) + "\n")